In [38]:
### Imports

from pathlib import Path
import gradio as gr
from openai import OpenAI
from IPython.display import Markdown
from typing import Any

In [39]:
### LLM setup

OLLAMA = {
    "BASE_URL": "http://localhost:11434/v1",
    "MODEL": "gemma4",
    "API_KEY": "ollama",
}

url, model, apikey = OLLAMA.get("BASE_URL"), OLLAMA.get("MODEL"), OLLAMA.get("API_KEY")

ollama = OpenAI(base_url=url, api_key=apikey)

In [40]:
### Read all employee data into a dictionary

KNOWLEDGE_BASE = Path("../knowledge-base")

knowledge: dict[str, str] = {}

employees = (KNOWLEDGE_BASE / "employees").glob("*")


for filename in employees:
    name = Path(filename).stem.split(' ')[-1]
    with open (filename, "r", encoding="utf-8") as f:
        knowledge[name.lower()] = f.read()

In [ ]:
knowledge

In [ ]:
display(Markdown(knowledge.get("lancaster")))

In [43]:
### Same thing but for products

products = (KNOWLEDGE_BASE / "products").glob("*")

for filename in products:
    name = Path(filename).stem.split(' ')[-1]
    with open (filename, "r", encoding="utf-8") as f:
        knowledge.setdefault(name.lower(), f.read())

In [44]:
knowledge.keys()

dict_keys(['chen', 'harper', 'thomson', 'foster', 'lancaster', 'walker', 'rodriguez', 'park', 'kim', 'carter', 'tran', 'wilson', 'adams', 'liu', 'blake', 'bishop', 'zhang', 'anderson', 'johnson', 'thompson', "o'brien", 'rivera', 'patel', 'spencer', 'sharma', 'martinez', 'greene', 'trenton', 'williams', 'brooks', 'bizllm', 'carllm', 'claimllm', 'healthllm', 'homellm', 'lifellm', 'markellm', 'rellm'])

In [45]:
SYSTEM_PREFIX = """
You represent Insurellm, the Insurance Tech company.
You are an expert in answering questions about Insurellm; its employees and its products.
You are provided with additional context that might be relevant to the user's question.
Give brief, accurate answers. If you don't know the answer, say so.

Relevant context:
"""

In [ ]:
### Internal helper to fetch the relevant context

def get_relevant_context(message: Any) -> list[str]:
    text = ''.join(ch for ch in message if ch.isalpha() or ch.isspace())
    words = text.lower().split()
    return [knowledge.get(word) for word in words if word in knowledge]

In [ ]:
get_relevant_context("Who is Lancaster and what is carllm?")

In [ ]:
### Helper to use context to enrich prompt

def get_additional_context(message: Any) -> str:
    res = get_relevant_context(message)
    if not res:
        return "No additional context relevant to the user's question.."
    return f"The following additional context might be relevant in answering the user's question:\n\n" + "\n\n".join(res)

In [ ]:
### Chat function

def chat(message: Any, history: dict) -> str:
    sys_msg = SYSTEM_PREFIX + get_additional_context(message)
    messages = [{"role": "system", "content": sys_msg}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [50]:
### Gradio UI

gr.ChatInterface(fn=chat, type="messages").launch(inbrowser=True)

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.
